# 122 -- Structured Path A: Behavior-Only Phenotyping (Layer 2 analog)

**Blueprint:** `outputs/reports/structured/structured_path_a_blueprint.md` -- read this first.
**Exploration plan:** `outputs/reports/structured/structured_step2_exploration_plan.md`
**Mirrors:** DS5's `notebooks/102_ds5_behavior_only_phenotype.ipynb` (Path A), adapted -- same question
(what phenotype structure emerges when NeT/network features are excluded?), different population and
feature set (Structured's own 6 non-NeT curated Step 1 features, not DS5's sig_hat/H_emp/x_mean).

**Output root:** `outputs/model_outputs/structured/phenotyping/path_a/`


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score
from sklearn.decomposition import PCA
from scipy.optimize import linear_sum_assignment
import warnings
warnings.filterwarnings("ignore")

MO = PROJECT_ROOT / "outputs" / "model_outputs" / "structured"
FEATURES = PROJECT_ROOT / "outputs" / "features" / "structured"
OUT = MO / "phenotyping" / "path_a"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
print("Output folder:", OUT)

## 1. Population (Part 1 of blueprint) -- complete-case across 6 non-NeT eligibility flags

`is_net_eligible` is deliberately excluded from the intersection -- that is the entire point of Path A.

In [ ]:
net = pd.read_csv(MO / "network_theory" / "participant_network_metrics.csv")
intt = pd.read_csv(MO / "information_theory" / "participant_entropy_metrics.csv")
stop = pd.read_csv(MO / "stochastic_processes" / "participant_stop_parameters.csv")
dys = pd.read_csv(MO / "dynamical_systems" / "participant_trajectory_metrics.csv")
syd = pd.read_csv(MO / "system_dynamics" / "participant_syd_summary.csv")
stm = pd.read_csv(MO / "statistical_mechanics" / "participant_stm_summary.csv")
gat = pd.read_csv(MO / "game_theory" / "participant_gat_summary.csv")
part_feat = pd.read_csv(FEATURES / "structured_participant_features.csv")
phase1 = pd.read_csv(MO / "phenotyping" / "cluster_assignments.csv")

FEATURE_MAP = {
    "H_completion": intt[["participant_id", "H_completion"]],
    "created_Fano_factor": stop[stop["created_is_eligible"]][["participant_id", "created_Fano_factor"]],
    "state_space_volatility": dys[dys["is_dys_eligible"]][["participant_id", "state_space_volatility"]],
    "backlog_trend_slope": syd[syd["is_syd_eligible"]][["participant_id", "backlog_trend_slope"]],
    "susceptibility": stm[stm["is_stm_eligible"]][["participant_id", "susceptibility"]],
    "mean_commitment_score": gat[gat["is_gat_eligible"]][["participant_id", "mean_commitment_score"]],
}
feature_cols = list(FEATURE_MAP.keys())

feat_df = None
for col, df in FEATURE_MAP.items():
    feat_df = df if feat_df is None else feat_df.merge(df, on="participant_id", how="outer")

complete_case = feat_df.dropna(subset=feature_cols).reset_index(drop=True)
print(f"Path A complete-case population (6 non-NeT features): N={len(complete_case)}")
print(f"Phase 1 (7-feature, incl. NeT) population was N={len(phase1)}")
print(f"Gain from dropping NeT requirement: {len(complete_case) - len(set(complete_case.participant_id) & set(phase1.participant_id))} new participants" if False else "")
assert len(complete_case) == 41, f"expected N=41 per blueprint verification, got {len(complete_case)}"
complete_case.head()

## 2. Feature redundancy check (carried forward from Phase 1, not re-litigated)

In [ ]:
corr_matrix = complete_case[feature_cols].corr()
print(corr_matrix.round(3))

flagged = []
for i, c1 in enumerate(feature_cols):
    for c2 in feature_cols[i+1:]:
        r = corr_matrix.loc[c1, c2]
        if abs(r) > 0.6:
            flagged.append((c1, c2, round(r, 3)))

print()
if flagged:
    print("Pairs with |r| > 0.6:")
    for c1, c2, r in flagged:
        print(f"  {c1} <-> {c2}: r={r}  (H_completion/state_space_volatility was already flagged at r=0.68 in Phase 1's N=36 -- consistent, not new)")
else:
    print("No pairs with |r| > 0.6.")

complete_case.to_csv(OUT / "behavior_only_feature_matrix.csv", index=False)
print(f"\nSaved: behavior_only_feature_matrix.csv {complete_case.shape}")

## 3. K-selection (Part 3 of blueprint)

k=2..4 (k=4 tested as the practical ceiling given N=41 and min_cell>=10), same procedure as Phase 1.

In [ ]:
X = complete_case[feature_cols].to_numpy()
X_z = (X - X.mean(axis=0)) / X.std(axis=0)

MIN_CELL = 10
k_results = {}
for k in range(2, 5):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_z)
    labels = km.labels_
    sizes = pd.Series(labels).value_counts()
    sil = silhouette_score(X_z, labels)
    ch = calinski_harabasz_score(X_z, labels)
    db = davies_bouldin_score(X_z, labels)
    k_results[k] = {"labels": labels, "silhouette": sil, "CH": ch, "DB": db,
                     "min_size": int(sizes.min()), "sizes": sizes.to_dict()}
    print(f"k={k}: silhouette={sil:.3f}, CH={ch:.1f}, DB={db:.3f}, min_cell={sizes.min()}, sizes={dict(sizes)}")

valid_ks = [k for k, r in k_results.items() if r["min_size"] >= MIN_CELL]
print(f"\nValid k (min_cell >= {MIN_CELL}): {valid_ks}")
SELECTED_K = max(valid_ks, key=lambda k: k_results[k]["silhouette"]) if valid_ks else 2
print(f"Selected k: {SELECTED_K}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
ks = list(k_results.keys())
axes[0].plot(ks, [k_results[k]["silhouette"] for k in ks], marker="o")
axes[0].set_title("Silhouette (higher better)"); axes[0].set_xlabel("k")
axes[1].plot(ks, [k_results[k]["CH"] for k in ks], marker="o", color="#55A868")
axes[1].set_title("Calinski-Harabasz (higher better)"); axes[1].set_xlabel("k")
axes[2].plot(ks, [k_results[k]["DB"] for k in ks], marker="o", color="#C44E52")
axes[2].set_title("Davies-Bouldin (lower better)"); axes[2].set_xlabel("k")
for ax in axes:
    ax.axvline(SELECTED_K, color="gray", ls="--", lw=1)
plt.tight_layout()
plt.savefig(FIG / "fig1_patha_k_selection.png", dpi=150)
plt.show()
print("Saved fig1_patha_k_selection.png")

## 4. Bootstrap Jaccard stability

In [ ]:
def bootstrap_jaccard(X_z, labels_full, k, n_boot=100, frac=0.8, seed=RANDOM_STATE):
    n = len(X_z)
    rng = np.random.RandomState(seed)
    jaccards = []
    for b in range(n_boot):
        idx = rng.choice(n, size=int(n * frac), replace=False)
        km_sub = KMeans(n_clusters=k, n_init=10, random_state=seed + b).fit(X_z[idx])
        labels_sub = km_sub.labels_
        labels_full_sub = labels_full[idx]
        cont = np.zeros((k, k))
        for i in range(k):
            for j in range(k):
                cont[i, j] = np.sum((labels_full_sub == i) & (labels_sub == j))
        row_ind, col_ind = linear_sum_assignment(-cont)
        pair_jaccards = []
        for i, j in zip(row_ind, col_ind):
            set_full = set(np.where(labels_full_sub == i)[0])
            set_sub = set(np.where(labels_sub == j)[0])
            union = len(set_full | set_sub)
            pair_jaccards.append(len(set_full & set_sub) / union if union > 0 else np.nan)
        jaccards.append(np.nanmean(pair_jaccards))
    return float(np.mean(jaccards)), float(np.std(jaccards))

jac_mean, jac_std = bootstrap_jaccard(X_z, k_results[SELECTED_K]["labels"], SELECTED_K)
stability_verdict = "STABLE" if jac_mean >= 0.75 else "MARGINAL" if jac_mean >= 0.60 else "UNSTABLE"
print(f"Bootstrap Jaccard (k={SELECTED_K}, 100 resamples, 80% subsample): mean={jac_mean:.3f}, std={jac_std:.3f}")
print(f"Stability verdict: {stability_verdict}")
print("(N=41 caveat: noisier than DS5's Path A at N=414)")

## 5. PCA sanity check

In [ ]:
pca = PCA(n_components=len(feature_cols))
pca.fit(X_z)
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {v*100:.1f}%")
print(f"Cumulative PC1+PC2: {pca.explained_variance_ratio_[:2].sum()*100:.1f}%")

fig, ax = plt.subplots(figsize=(7, 6))
coords = pca.transform(X_z)[:, :2]
labels_sel = k_results[SELECTED_K]["labels"]
for lab in np.unique(labels_sel):
    mask = labels_sel == lab
    ax.scatter(coords[mask, 0], coords[mask, 1], label=f"Cluster {lab} (n={mask.sum()})", alpha=0.7)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title(f"Path A PCA biplot, k={SELECTED_K}")
ax.legend()
plt.tight_layout()
plt.savefig(FIG / "fig2_patha_pca_biplot.png", dpi=150)
plt.show()
print("Saved fig2_patha_pca_biplot.png")

## 6. Cluster assignments, profiles, and comparison figure

In [ ]:
PROFILE_NAMES = [f"BehavProfile {chr(65+i)}" for i in range(SELECTED_K)]
label_to_name = {lab: PROFILE_NAMES[i] for i, lab in enumerate(sorted(np.unique(labels_sel)))}
complete_case["cluster_label"] = [label_to_name[l] for l in labels_sel]

cluster_assignments = complete_case[["participant_id", "cluster_label"] + feature_cols].copy()
cluster_assignments.to_csv(OUT / "behavior_only_cluster_assignments.csv", index=False)

cluster_profiles = complete_case.groupby("cluster_label")[feature_cols].agg(["mean", "median", "std"])
cluster_profiles.to_csv(OUT / "behavior_only_cluster_profiles.csv")

print("Cluster sizes:")
print(complete_case["cluster_label"].value_counts())
print()
print("Cluster means (raw):")
print(complete_case.groupby("cluster_label")[feature_cols].mean().round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
means = complete_case.groupby("cluster_label")[feature_cols].mean()
means_z = (means - complete_case[feature_cols].mean()) / complete_case[feature_cols].std()
means_z.T.plot(kind="bar", ax=ax)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("z-score relative to Path A population mean")
ax.set_title(f"Path A (behavior-only) cluster profiles, k={SELECTED_K}, N={len(complete_case)}")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIG / "fig3_patha_cluster_profiles.png", dpi=150)
plt.show()
print("Saved fig3_patha_cluster_profiles.png")

## 7. Comparison to Phase 1 (Part 4 of blueprint -- the falsifiable prediction)

**Prediction stated in the blueprint before running:** Structured's Phase 1 split is only weakly
NeT-driven, so Path A (which excludes NeT entirely) should reproduce Phase 1 with HIGHER ARI than DS5's
near-zero (0.020) -- the opposite pattern from DS5, where Phase 1 was NeT-dominated and Path A was
orthogonal to it.

In [ ]:
overlap = complete_case.merge(phase1[["participant_id", "cluster_label"]], on="participant_id",
                               suffixes=("_patha", "_phase1"))
print(f"Overlap N (in both Path A and Phase 1): {len(overlap)}")

ari = adjusted_rand_score(overlap["cluster_label_phase1"], overlap["cluster_label_patha"])
print(f"ARI(Path A vs Phase 1) on overlap N={len(overlap)}: {ari:.3f}")
print(f"DS5's equivalent (Path A vs its Phase 1): 0.020 (orthogonal)")

if ari > 0.3:
    verdict = "CONFIRMED: Path A substantially reproduces Phase 1 -- Structured's dominant split IS the behavioral-dynamics axis, unlike DS5 where it was network-driven."
elif ari > 0.1:
    verdict = "PARTIAL: some correspondence but not strong reproduction -- more nuanced than a single dominant axis."
else:
    verdict = "NOT CONFIRMED: Path A is roughly as orthogonal to Phase 1 as DS5's was -- the pre-stated reasoning (Phase1 gap on HHI_net being small) did not translate into cluster-label correspondence. Reported as a genuine surprise, not reframed."
print()
print(verdict)

cross_tab = pd.crosstab(overlap["cluster_label_phase1"], overlap["cluster_label_patha"])
print()
print("Cross-tabulation (Phase 1 rows x Path A columns):")
print(cross_tab)

ari_result = pd.DataFrame([{"comparison": "Path A vs Phase 1", "overlap_n": len(overlap),
                             "ARI": round(ari, 4), "ds5_equivalent_ARI": 0.020, "verdict": verdict}])
ari_result.to_csv(OUT / "behavior_only_vs_phase1_ari.csv", index=False)
print("\nSaved: behavior_only_vs_phase1_ari.csv")

## 8. Path E analog: subscription-stratified feasibility check (Part 5 of blueprint)

DS5's Path E used three randomized friction conditions. Structured has none -- the only available
stratifier is the observational `subscription_type_last` (free/pro). **Prediction stated in advance:**
the free subgroup (N=12) is very likely too small for a stable k=2 split.

In [ ]:
sub = part_feat.set_index("participant_id")["subscription_type_last"]
complete_case["subscription_type_last"] = complete_case["participant_id"].map(sub)
print(complete_case["subscription_type_last"].value_counts())

sub_results = []
for grp_name, grp_df in complete_case.groupby("subscription_type_last"):
    n = len(grp_df)
    if n < 6:  # hard floor: k=2 cannot even nominally give both cells >=3 below this
        sub_results.append({"subgroup": grp_name, "n": n, "feasible": False,
                             "reason": f"N={n} < 6 -- too small to attempt k=2 at all",
                             "jaccard_mean": np.nan})
        print(f"{grp_name} (N={n}): SKIPPED -- below the N=6 hard floor for any k=2 attempt")
        continue
    Xg = grp_df[feature_cols].to_numpy()
    Xg_z = (Xg - Xg.mean(axis=0)) / Xg.std(axis=0)
    km_g = KMeans(n_clusters=2, n_init=10, random_state=RANDOM_STATE).fit(Xg_z)
    sizes_g = pd.Series(km_g.labels_).value_counts()
    if sizes_g.min() < MIN_CELL:
        sub_results.append({"subgroup": grp_name, "n": n, "feasible": False,
                             "reason": f"k=2 ran, sizes={dict(sizes_g)}, min_cell={sizes_g.min()} < {MIN_CELL} "
                                       f"(same floor used for k-selection everywhere else in this notebook)",
                             "jaccard_mean": np.nan})
        print(f"{grp_name} (N={n}): k=2 ran, sizes={dict(sizes_g)} -- INFEASIBLE (min_cell < {MIN_CELL})")
        continue
    jm, js = bootstrap_jaccard(Xg_z, km_g.labels_, 2, n_boot=100, frac=0.8, seed=RANDOM_STATE)
    sub_results.append({"subgroup": grp_name, "n": n, "feasible": True,
                         "reason": f"k=2 ran, sizes={dict(sizes_g)}, min_cell={sizes_g.min()} >= {MIN_CELL}",
                         "jaccard_mean": round(jm, 3)})
    print(f"{grp_name} (N={n}): k=2 ran, sizes={dict(sizes_g)}, bootstrap Jaccard={jm:.3f} "
          f"({'STABLE' if jm>=0.75 else 'unstable/marginal, N too small to trust'})")

sub_result_df = pd.DataFrame(sub_results)
sub_result_df.to_csv(OUT / "subscription_stratified_check.csv", index=False)
print()
print(sub_result_df.to_string(index=False))
print("\nSaved: subscription_stratified_check.csv")

## 9. model_run_summary.md

In [ ]:
summary = f"""# Structured Path A -- Behavior-Only Phenotyping -- Model Run Summary

**Run date:** 2026-08-05
**Notebook:** notebooks/122_structured_behavior_only_phenotype.ipynb
**Blueprint:** outputs/reports/structured/structured_path_a_blueprint.md

## Population
- Path A complete-case (6 non-NeT features): N={len(complete_case)}
- Phase 1 (7-feature) population was N={len(phase1)}

## Feature redundancy
{"No pairs |r|>0.6." if not flagged else str(flagged) + " (H_completion/state_space_volatility carried forward from Phase 1, not new)"}

## K-selection
"""
for k in k_results:
    r = k_results[k]
    summary += f"- k={k}: silhouette={r['silhouette']:.3f}, CH={r['CH']:.1f}, DB={r['DB']:.3f}, min_cell={r['min_size']}, sizes={r['sizes']}\n"
summary += f"""
**Selected k={SELECTED_K}** (silhouette={k_results[SELECTED_K]['silhouette']:.3f}, subject to min_cell>={MIN_CELL})

## Stability
- Bootstrap Jaccard (100 resamples, 80% subsample): mean={jac_mean:.3f}, std={jac_std:.3f}
- Verdict: {stability_verdict}

## PCA
- PC1+PC2 variance explained: {pca.explained_variance_ratio_[:2].sum()*100:.1f}%

## Cluster profiles (raw means)
{complete_case.groupby("cluster_label")[feature_cols].mean().round(4).to_string()}

## Cluster sizes
{complete_case["cluster_label"].value_counts().to_string()}

## Comparison to Phase 1 (the central Path A question)
- Overlap N: {len(overlap)}
- ARI(Path A vs Phase 1): {ari:.3f}  (DS5's equivalent: 0.020, orthogonal)
- {verdict}

## Path E analog: subscription-stratified feasibility check
{sub_result_df.to_string(index=False)}

## Key limitations
1. N=41 -- much smaller than DS5's Path A (N=414); all statistics noisier
2. Same H_completion/state_space_volatility correlation (r~0.63-0.68) present as in Phase 1, not resolved
3. Path E analog is a weak, observational (non-randomized) stratifier -- not a real test of condition-invariance
4. k=4 tested but likely infeasible at this N; k selection constrained by min_cell=10 floor
"""

with open(OUT / "model_run_summary.md", "w") as f:
    f.write(summary)

print(summary)